# GENESIS — End-to-End GRPO Training on a Multi-Agent Startup Environment

**OpenEnv Hackathon (India 2026) submission notebook.**

[![OpenEnv](https://img.shields.io/badge/OpenEnv-compatible-6366f1)](https://huggingface.co/openenv) [![HF Space](https://img.shields.io/badge/Space-genesis__env-blue)](https://huggingface.co/spaces/rhine-pereira/genesis_env)

This notebook is the **canonical reproduction recipe** for the GENESIS submission. It runs end-to-end on a free Colab T4 in roughly 10–20 minutes and produces:

1. A live, OpenEnv-compliant **MCP server** (`server.app:app` on port 7860).
2. A **GRPO** (TRL) training run against that server, with the GENESIS 11-component composable rubric as the reward signal.
3. A **training-evidence plot** (`outputs/evals/training_progress.png`) that judges can inspect directly without re-running anything.
4. A **reward-curve plot + summary JSON** (`outputs/evals/reward_curves.png`, `outputs/evals/reward_summary.json`) computed from the persisted session log.
5. A saved **LoRA adapter** under `./genesis-checkpoints/final/` ready to be uploaded to a HF Hub model card.

## What you get when you run this notebook

| Stage | Cell | What happens |
|------|------|--------------|
| 0 | GPU + runtime check | Verifies a GPU is attached (T4 ≥ 15 GB recommended) |
| 1 | Clone repo | `git clone https://github.com/rhine-pereira/meta-hack-finale.git` |
| 2 | Install dependencies | OpenEnv core, FastMCP, TRL, PEFT, bitsandbytes, accelerate, datasets, matplotlib |
| 3 | Smoke test | Boots the server, runs 5 reward rollouts (no GPU needed) — proves env + reward pipeline works |
| 4 | GRPO training | Runs `train_colab.py`; trains Qwen2.5-3B-Instruct (4-bit + LoRA) for `--steps 60` |
| 5 | Render training-evidence plot | Reads `outputs/evals/training_log.jsonl` and plots reward + curriculum curves |
| 6 | Generate full reward curves | Runs `scripts/plot_rewards.py` on `sessions.pkl` for the README artifact |
| 7 | Display all plots inline | Final visual summary |
| 8 | Optional — upload artifacts to HF Hub | Push the LoRA adapter + plots |

## Submission requirements covered by this notebook

- ✅ **OpenEnv (latest release)** — `openenv-core[core]>=0.2.3` is installed in cell 2 and the server uses the standard FastMCP runtime.
- ✅ **HF TRL** — we use **TRL `GRPOTrainer`** with **bitsandbytes** 4-bit quantization + LoRA (free Colab T4).
- ✅ **HF Space hosting** — the same server (`server.app:app`) runs on [the Space](https://huggingface.co/spaces/rhine-pereira/genesis_env). This notebook just runs a local copy.
- ✅ **Reward + training evidence** — cells 5–7 produce `training_progress.png` and `reward_curves.png` automatically.
- ✅ **Mini-blog / video link** — see [`submission/hf_mini_blog.md`](https://github.com/rhine-pereira/meta-hack-finale/blob/main/submission/hf_mini_blog.md) and the demo script.

> ⏱ **Tip:** if you only have ~5 minutes, run cells 0–3 + the smoke cell. Skip cell 4 and re-render the existing committed plots (cell 7).  
> 🔧 **If something goes wrong:** scroll to the **Troubleshooting** cell at the bottom for the most common Colab pitfalls (Triton/PTX, OOM, BnB CUDA versions).

## 0. Runtime + GPU sanity check

Make sure your Colab is running on a GPU runtime:

**Runtime → Change runtime type → Hardware accelerator: T4 GPU** (or A100 if you have Pro).

In [ ]:
import platform, sys, subprocess, os
print(f"Python  : {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"In Colab: {'google.colab' in sys.modules or os.path.exists('/content')}")

try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], text=True).strip()
    print("\nGPU detected:")
    print(" ", out)
except Exception as e:
    print("\n⚠️  No GPU detected. The smoke test will still work, but training requires a GPU.")
    print("   Switch runtime: Runtime → Change runtime type → T4 GPU.")

## 1. Clone the GENESIS repository

We clone the public mirror so all source files (`server/`, `train_colab.py`, `scripts/plot_rewards.py`, etc.) are available on the Colab VM.

In [ ]:
import os, subprocess, sys

REPO_URL  = "https://github.com/rhine-pereira/meta-hack-finale.git"
REPO_DIR  = "/content/meta-hack-finale" if os.path.exists("/content") else os.path.abspath("meta-hack-finale")

if not os.path.isdir(REPO_DIR):
    print(f"Cloning → {REPO_DIR}")
    subprocess.check_call(["git", "clone", "--depth=1", REPO_URL, REPO_DIR])
else:
    print(f"Repo present, pulling latest → {REPO_DIR}")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("\ncwd =", os.getcwd())
print("Top-level files:", [p for p in os.listdir('.') if not p.startswith('.')][:25])

## 2. Install dependencies

We pin **`torch`** to whatever Colab pre-installed — upgrading torch usually drags in a Triton that emits PTX 8.7, which Colab's T4 ptxas can't consume.

> Expect ~3 minutes the first time. If Colab asks to **restart the runtime** after this cell, restart and re-run from cell 1.

In [ ]:
import sys, subprocess, importlib.metadata

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    torch_version = importlib.metadata.version("torch")
    torch_pin = f"torch=={torch_version}"
    print(f"Pinning torch to {torch_pin} (already installed by Colab).")
except Exception:
    torch_pin = "torch"
    print("No pre-existing torch detected; will install latest.")

print("\n[1/2] OpenEnv + server stack…")
pip("openenv-core[core]>=0.2.3", "fastapi", "uvicorn[standard]", "fastmcp",
    "requests", "numpy", "matplotlib", "pillow", "base58")

print("[2/2] HF training stack (TRL / PEFT / bitsandbytes / accelerate / datasets)…")
pip(torch_pin, "trl>=0.11.0", "peft>=0.12.0", "bitsandbytes>=0.43.0",
    "accelerate>=0.34.0", "datasets>=2.20.0", "transformers>=4.44.0")

print("\nDone. Quick import check:")
import torch, transformers, trl, peft, bitsandbytes, datasets
print("  torch       ", torch.__version__, "cuda=", torch.cuda.is_available())
print("  transformers", transformers.__version__)
print("  trl         ", trl.__version__)
print("  peft        ", peft.__version__)
print("  bitsandbytes", bitsandbytes.__version__)
print("  datasets    ", datasets.__version__)

## 3. Smoke test — verify the OpenEnv server + reward pipeline

This boots the **GENESIS MCP server** in a subprocess, runs **5 reward rollouts** (one per role), and exits. **No GPU is needed** — it proves the environment, MCP wiring, and reward function are healthy.

Look for `Smoke test passed.` at the bottom of the output.

In [ ]:
import os
os.environ["PYTHONPATH"] = os.getcwd()
!PYTHONPATH=. python train_colab.py --smoke

## 4. GRPO training — Qwen2.5-3B-Instruct (4-bit + LoRA)

Now the main event. This cell:

1. Boots the OpenEnv server (background subprocess).
2. Loads `Qwen/Qwen2.5-3B-Instruct` with **bitsandbytes** 4-bit quantization.
3. Wraps it with **LoRA** adapters (`r=16`, `q/k/v/o_proj`).
4. Runs **TRL `GRPOTrainer`** for `--steps 60` with `--num-generations 2` (the GRPO minimum) and the GENESIS reward function as the only reward.
5. Logs each reward-fn call to `outputs/evals/training_log.jsonl`.
6. Saves the adapter to `./genesis-checkpoints/final/` and renders `outputs/evals/training_progress.png`.

### Knobs worth tweaking

| Flag | Default | Use when |
|------|---------|----------|
| `--steps` | `60` | Increase to 200+ for stronger evidence on A100 / longer runs |
| `--episode-days` | `20` | Longer episodes give the reward more signal but slow training |
| `--difficulty` | `1` | Set higher to start in a harder MarketMaker regime |
| `--num-generations` | `2` | GRPO minimum. 4 is better on A100 |
| `--max-completion-length` | `160` | Lower for speed, higher for richer brain entries |
| `--fast` | off | Aggressive speed profile for ~5-min Colab runs |

> ⏱ Expected runtime on a free Colab T4 with the defaults below: **~10–18 minutes**.  
> 💾 Memory: ~9–11 GB VRAM.

In [ ]:
!PYTHONPATH=. python train_colab.py \
    --steps 600 \
    --episode-days 30 \
    --difficulty 1 \
    --num-generations 2 \
    --max-completion-length 160 \
    --model Qwen/Qwen2.5-3B-Instruct \
    --output ./genesis-checkpoints \
    --log-dir outputs/evals

## 5. Inline training-evidence plot

`train_colab.py` already wrote `outputs/evals/training_progress.png` at the end of training. Display it here so judges see the learning trajectory + curriculum without leaving the notebook.

**What to look for:**
- Left panel: per-call reward (translucent dots) and a moving average that should trend up (or, in a short run, at minimum stay clustered above the baseline ~0.40).
- Right panel: MarketMaker difficulty. If the model improves enough, you'll see ⬆ promotions to L2 / L3.

In [ ]:
import os
from PIL import Image
from IPython.display import display, Markdown

p = "outputs/evals/training_progress.png"
if os.path.exists(p):
    display(Markdown(f"**{p}**"))
    display(Image.open(p))
else:
    print("⚠️  No training plot found yet. Re-run cell 4 to generate it.")

## 6. Cross-model reward curves (the README artifact)

We additionally ship `scripts/plot_rewards.py`, which renders **per-episode reward curves grouped by `model_id`** from the persisted MCP session pickle. This is the figure embedded in the [README](https://github.com/rhine-pereira/meta-hack-finale#evaluation-evidence-model-wise-figure).

It draws three things:
1. **Episode reward curves** with mean ± std bands (one curve per model).
2. **Final-reward scatter** colored by model.
3. A **summary JSON** with per-model averages.

In [ ]:
import os
if os.path.exists('sessions.pkl'):
    !PYTHONPATH=. python scripts/plot_rewards.py --sessions sessions.pkl --out outputs/evals --summarize-models
else:
    print("sessions.pkl not produced by training (older server versions). Falling back to demo plot.")
    !PYTHONPATH=. python scripts/plot_rewards.py --demo --out outputs/evals

## 7. Display all generated artifacts

In [ ]:
import os, json
from PIL import Image
from IPython.display import display, Markdown

for path in ["outputs/evals/training_progress.png", "outputs/evals/reward_curves.png"]:
    if os.path.exists(path):
        display(Markdown(f"### `{path}`"))
        display(Image.open(path))
    else:
        display(Markdown(f"_Missing: `{path}`_"))

summary_path = "outputs/evals/reward_summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    display(Markdown("### Reward summary (`outputs/evals/reward_summary.json`)"))
    display(Markdown("```json\n" + json.dumps(summary, indent=2) + "\n```"))
else:
    display(Markdown("_No reward_summary.json found._"))

## 8. (Optional) Push artifacts to Hugging Face Hub

If you want to publish the trained adapter alongside the GENESIS Space, uncomment the cell below and provide an HF token with **write** access. This pushes:

- The LoRA adapter (`./genesis-checkpoints/final/`) → `<your-user>/genesis-grpo-qwen2.5-1.5b`
- The training-evidence plot to the same repo as a README banner.

> Skip this if you only want the local artifacts.

In [ ]:
# from huggingface_hub import HfApi, login, create_repo
#
# HF_TOKEN = "hf_..."            # ← paste a write-token here
# REPO_ID  = "your-username/genesis-grpo-qwen2.5-1.5b"
#
# login(token=HF_TOKEN)
# create_repo(REPO_ID, exist_ok=True, repo_type="model")
# api = HfApi()
# api.upload_folder(folder_path="./genesis-checkpoints/final", repo_id=REPO_ID, repo_type="model")
# api.upload_file(
#     path_or_fileobj="outputs/evals/training_progress.png",
#     path_in_repo="training_progress.png",
#     repo_id=REPO_ID,
#     repo_type="model",
# )
# print(f"✅ pushed → https://huggingface.co/{REPO_ID}")

## 9. Troubleshooting

**`ModuleNotFoundError: triton.backends`** — this happens when `torch.compile` / `torch._dynamo` paths get pulled in. `train_colab.py` already disables them (`TORCHDYNAMO_DISABLE=1` etc.) but if you import torch in a **different** kernel before running the script, restart the runtime.

**`PTXASError: PTX 8.7 unsupported`** — your torch was upgraded after Colab boot, dragging Triton with it. Re-run cell 2 — it pins torch to Colab's pre-installed version.

**`CUDA OOM`** — drop to `--max-completion-length 96 --num-generations 2 --fast`, or switch the model to `Qwen/Qwen2.5-1.5B-Instruct`.

**Reward stays at exactly `0.0`** — the model is emitting completions the parser can't read. Inspect `outputs/evals/training_log.jsonl` and check the `completion_preview` column. The reward-fn falls back to `make_decision(plain text)` if JSON parsing fails, so a flat 0 usually means the MCP server itself crashed (`tail /tmp/genesis_server.log`).

**Reward never crosses 0.5** — that's expected for a 60-step run on Qwen2.5-3B; the dynamic range of GENESIS only saturates after a few hundred steps with multiple generations. Bump `--steps 200 --num-generations 4` on an A100.

**Server didn't start in 30 s** — check `/tmp/genesis_server.log`. Most common cause is a stale port. Restart the kernel.

## 10. Where to go next

- **Hugging Face Space (live env):** [`rhine-pereira/genesis_env`](https://huggingface.co/spaces/rhine-pereira/genesis_env)
- **Mini-blog:** [`submission/hf_mini_blog.md`](https://github.com/rhine-pereira/meta-hack-finale/blob/main/submission/hf_mini_blog.md)
- **Demo video script:** [`submission/demo_video_script.md`](https://github.com/rhine-pereira/meta-hack-finale/blob/main/submission/demo_video_script.md)
- **Reward design doc:** [`server/reward_engine.py`](https://github.com/rhine-pereira/meta-hack-finale/blob/main/server/reward_engine.py)
- **Full design narrative:** [`context/hackathon_idea_3.md`](https://github.com/rhine-pereira/meta-hack-finale/blob/main/context/hackathon_idea_3.md)